In [1]:
import os
import json
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO


os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
# CUDA Configuration
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
model = YOLO('yolov8s-seg.pt') 
#File of the dataset
data_yaml = 'dataset.yaml' 

100%|██████████| 22.8M/22.8M [00:02<00:00, 10.2MB/s]


In [4]:
try:
    # Entrena el modelo
    model.train(
        data=data_yaml,  # Ruta al archivo YAML de configuración
        epochs=100,                # Número de épocas
        imgsz=640,
        batch=12,  
        device='cuda:0',   
        workers=0,
        project="Result_trained",
        name="Bubbles_dectection",           # Tamaño de las imágenes
    )
except RuntimeError as e:
    print(f"Error durante el entrenamiento: {e}")
print("Entrenamiento completado")

New https://pypi.org/project/ultralytics/8.3.61 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.28  Python-3.11.9 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=segment, mode=train, model=yolov8s-seg.pt, data=dataset.yaml, epochs=100, time=None, patience=100, batch=12, imgsz=640, save=True, save_period=-1, cache=False, device=cuda:0, workers=0, project=Result_trained, name=Bubbles_dectection, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=Fals

train: Scanning C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\train\labels... 0 images, 68 backgrounds, 0 corrupt: 100%|██████████| 68/68 [00:00<00:00, 392.96it/s]

train: WARNING  No labels found in C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\train\labels.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
train: New cache created: C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\train\labels.cache
WARNING  No labels found in C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\train\labels.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.



val: Scanning C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\val\labels... 0 images, 18 backgrounds, 0 corrupt: 100%|██████████| 18/18 [00:00<00:00, 363.41it/s]


val: WARNING  No labels found in C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\val\labels.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
val: New cache created: C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\val\labels.cache
WARNING  No labels found in C:\Users\yourk\Documents\3.Universitat\Estadia\Clenead_Images\PU122\XY_clean\val\labels.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
Plotting labels to Result_trained\Bubbles_dectection\labels.jpg... 
zero-size array to reduction operation maximum which has no identity
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.00046875), 76 bias(decay=0.0)
TensorBoard: model graph visu

      1/100      3.65G          0          0      399.1          0          0        640: 100%|██████████| 6/6 [00:09<00:00,  1.56s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]

                   all         18          0          0          0          0          0          0          0          0          0
WARNING  no labels found in segment set, can not compute metrics without labels



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      2/100      3.48G          0          0        446          0          0        640:  67%|██████▋   | 4/6 [00:04<00:02,  1.06s/it]


KeyboardInterrupt: 

In [5]:
import detectron2
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances


ModuleNotFoundError: No module named 'detectron2'

In [ ]:

# Registra tu dataset en formato COCO
register_coco_instances("my_dataset_train", {}, "path/to/instances_train.json", "path/to/train/images")
register_coco_instances("my_dataset_val", {}, "path/to/instances_val.json", "path/to/val/images")

# Configura el modelo
cfg = get_cfg()
cfg.merge_from_file(detectron2.model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = detectron2.model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Cambia según el número de clases

# Entrena el modelo
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()